# Predicting the labour force status of young Rwandans

**UWENAYO Alain Pacifique — Machine Learning summative, African Leadership University**

| | |
|---|---|
| GitHub | https://github.com/uwenayoallain/rwanda-youth-labour-status-ml |
| Google Drive (raw data, models, logs, figures) | https://drive.google.com/drive/folders/1Ol9Xh2oEHdOokqBWSNi8RFORV1d6g2q_?usp=sharing |
| Video | _link to be added_ |

## The problem

Young Rwandans (16–30) have an unemployment rate of 18.5%, against 12.3% for adults, and more than a
quarter are not in work, school or training (NISR, 2024). My mission is to help students in Rwanda start
their careers, so I want to know which young people end up **employed**, which are **unemployed**
(looking for work), and which are **outside the labour force** (studying, caring for family, farming only
for the household, or discouraged) — and whether that can be predicted from things we know *before* the
outcome: education, vocational training, disability, migration, household role and unpaid work.

The data is the **Rwanda Labour Force Survey** (NISR), 2024 and 2025 rounds, 53,503 young people.
I compare two ways of modelling it:

1. **Traditional machine learning** (scikit-learn): logistic regression, random forest, XGBoost,
   LightGBM and an SVM, with hyper-parameter search and class weighting — 9 experiments.
2. **Deep learning** (TensorFlow/Keras): multilayer perceptrons built with the Functional API and fed
   through `tf.data`, varying regularisation, size, learning rate, optimiser, embeddings and class
   weights — 9 experiments.

Every run is logged to TensorBoard. CNNs are not used: they are built for images, and survey rows have
no spatial structure.

## 0. Setup

Runs locally or on Google Colab. On Colab the first cell mounts Drive, installs two packages and copies
the data from `My Drive/ALU_ML_Summative/data/`. At the end, all outputs are copied back to Drive.

In [ ]:
import os, sys, subprocess, shutil, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_PROJECT_DIR = "ALU_ML_Summative"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyreadstat", "tabulate"], check=True)
    drive_dir = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
    work = Path("/content/alu-summative-iml")
    (work / "notebooks").mkdir(parents=True, exist_ok=True)
    os.chdir(work / "notebooks")
    raw = Path("../data/raw"); raw.mkdir(parents=True, exist_ok=True)
    for z in sorted(drive_dir.glob("data/*.zip")):
        with zipfile.ZipFile(z) as zf:
            for member in zf.namelist():
                if member.endswith(".dta") and not (raw / Path(member).name).exists():
                    with zf.open(member) as src, open(raw / Path(member).name, "wb") as dst:
                        shutil.copyfileobj(src, dst)
    for f in sorted(drive_dir.glob("data/*.dta")):
        if not (raw / f.name).exists():
            shutil.copy(f, raw)
    print("Data ready:", sorted(p.name for p in raw.glob("*.dta")))

In [ ]:
import glob, json, random, re, time, warnings
from dataclasses import dataclass

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import seaborn as sns
import sklearn
import tensorflow as tf
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, auc, average_precision_score,
                             balanced_accuracy_score, classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, label_binarize
from sklearn.svm import SVC
from tensorflow import keras
from tensorflow.keras import layers
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight", "font.size": 9})
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("TensorFlow", tf.__version__, "| scikit-learn", sklearn.__version__)

### 0.1 Settings

Paths, the seed, who is in the sample, what we predict, and which survey variables are allowed as
inputs. The key rule: anything NISR uses to *build* the labour force status (hours worked, job search,
earnings, the main job, own-use farming) is off limits, otherwise the model would just copy the answer.

In [ ]:
RAW_DIR, PROCESSED_DIR = ROOT / "data" / "raw", ROOT / "data" / "processed"
MODELS_DIR, LOGS_DIR, FIGURES_DIR = ROOT / "models", ROOT / "logs", ROOT / "reports" / "figures"
EXPERIMENTS_CSV = ROOT / "reports" / "experiments.csv"
RAW_GLOB = "RW_LFS*.dta"
SEED = 42

AGE_COL, YOUTH_MIN_AGE, YOUTH_MAX_AGE = "A04", 16, 30
TARGET_COL = "status1"
TARGET_CODES = {1: 0, 2: 1, 3: 2}
TARGET_NAMES = {0: "Employed", 1: "Unemployed", 2: "Outside labour force"}
CLASS_LABELS = list(TARGET_NAMES.values())
SHORT_LABELS = ["Employed", "Unemployed", "Outside LF"]
N_CLASSES = 3

NUMERIC_FEATURES = ["A04", "A16Y", "A19", "B02B", "indb1", "hhsize",
                    "G01", "G02", "G03", "G04", "G05", "G06", "G07"]          # age, migration timing, schooling, household size, unpaid-work hours
ORDINAL_FEATURES = ["B02A", "attained", "A06", "A07", "A08", "A09", "A10", "A11", "TVT2"]   # education levels, disability items, TVET attainment
NOMINAL_FEATURES = ["province", "Code_UR", "code_dis", "A01", "A02", "A05", "A12", "A13", "A15",
                    "B01", "B07", "B08", "B09", "B10", "B13", "indb05", "EN01", "migrant", "head",
                    "disable", "age3_16_30", "LFS_year"]
LEAKY_FEATURES = ["C01", "C19"]        # paid work last week, looked for work last month: define the target, used only in the leakage test
WEIGHT_COLS = ["weight2"]
ALL_RAW_COLUMNS = sorted(set(NUMERIC_FEATURES + ORDINAL_FEATURES + NOMINAL_FEATURES + LEAKY_FEATURES + WEIGHT_COLS
                             + [TARGET_COL, "YA", "dG01", "dG02", "dG03", "dG04", "dG05", "dG06", "dG07", "TVT", "HHID"]))
SENTINEL_LABEL_PATTERN = r"^(missing|refused|don.?t know|do not know|not applicable|not stated|n/?a)$"
TEST_SIZE = VAL_SIZE = 0.15

### 0.2 Data preparation

Load every survey round in `data/raw/`, keep 16–30 year-olds, turn "Don't know" codes into missing
values using the labels stored in the Stata file, handle skip patterns, add a few engineered features,
split, and build the encoder.

In [ ]:
def load_raw(raw_dir=RAW_DIR):
    frames, meta = [], None
    for path in sorted(glob.glob(str(raw_dir / RAW_GLOB))):
        _, m = pyreadstat.read_dta(path, metadataonly=True)
        cols = [c for c in ALL_RAW_COLUMNS if c in m.column_names]
        df, meta = pyreadstat.read_dta(path, usecols=cols, apply_value_formats=False)
        df["source_file"] = Path(path).name
        if "LFS_year" not in df or df["LFS_year"].isna().all():
            df["LFS_year"] = int(re.search(r"(20\d{2})", path).group(1))
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No {RAW_GLOB} file in {raw_dir}")
    return pd.concat(frames, ignore_index=True), meta


def select_youth(df):
    keep = df[AGE_COL].between(YOUTH_MIN_AGE, YOUTH_MAX_AGE) & df[TARGET_COL].isin(TARGET_CODES)
    out = df.loc[keep].copy()
    out[TARGET_COL] = out[TARGET_COL].map(TARGET_CODES).astype(int)
    return out.reset_index(drop=True)


def clean_sentinels(df, meta):
    pattern = re.compile(SENTINEL_LABEL_PATTERN, re.IGNORECASE)
    out, report = df.copy(), {}
    for col in out.columns:
        labels = meta.variable_value_labels.get(col, {})
        codes = {float(v) for v, lab in labels.items() if pattern.match(str(lab).strip())}
        if col == AGE_COL:
            codes |= {98.0, 99.0}
        mask = out[col].isin(codes) if codes else None
        if mask is not None and mask.any():
            report[col] = int(mask.sum())
            out.loc[mask, col] = np.nan
    out.attrs["sentinel_report"] = report
    return out


def engineer_features(df):
    out = df.copy()
    flags = [f"dG0{k}" for k in range(1, 8)]
    for col in NUMERIC_FEATURES + flags + ["head", "disable", "TVT"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out[flags + ["head"]] = out[flags + ["head"]].fillna(0.0)          # blank flag means "no"
    for k in range(1, 8):                                             # blank hours with flag 0 means zero hours
        out[f"G0{k}"] = out[f"G0{k}"].where(out[f"G0{k}"].notna() | (out[f"dG0{k}"] != 0), 0.0)

    out["care_hours"] = out[["G06", "G07"]].sum(axis=1, min_count=1)
    out["unpaid_work_hours"] = out[[f"G0{k}" for k in range(1, 8)]].sum(axis=1, min_count=1)
    disability = out[["A06", "A07", "A08", "A09", "A10", "A11"]]
    out["any_disability"] = (disability.max(axis=1) >= 2).astype(float).where(disability.notna().any(axis=1))
    out["severe_disability"] = (disability.max(axis=1) >= 3).astype(float).where(disability.notna().any(axis=1))
    out["tertiary"] = (out["B02A"] == 6).astype(float).where(out["B02A"].notna())
    out["tvet_completed"] = out["B13"].isin([1, 2]).astype(float).where(out["B08"].notna())
    out["is_student"] = (out["B01"] == 1).astype(float).where(out["B01"].notna())
    out["schooling_gap"] = (out["A04"] - 6 - out["indb1"]).clip(lower=0)
    out["never_moved"] = (out["A15"] == 1).astype(float).where(out["A15"].notna())
    out["A16Y"] = out["A16Y"].where(out["A15"] != 1, 0.0)
    out["A19"] = out["A19"].where(out["A15"] != 1, 0.0)
    out["married_female"] = ((out["A01"] == 2) & out["A05"].isin([1, 2, 3])).astype(float).where(out[["A01", "A05"]].notna().all(axis=1))
    out["migrant"] = out["migrant"].fillna(0)
    return out


ENGINEERED = ["care_hours", "unpaid_work_hours", "any_disability", "severe_disability", "tertiary",
              "tvet_completed", "is_student", "schooling_gap", "never_moved", "married_female"]


@dataclass
class Split:
    X_train: pd.DataFrame; X_val: pd.DataFrame; X_test: pd.DataFrame
    y_train: np.ndarray; y_val: np.ndarray; y_test: np.ndarray

    def summary(self):
        rows = {}
        for name, y in [("train", self.y_train), ("val", self.y_val), ("test", self.y_test)]:
            counts = pd.Series(y).value_counts().sort_index()
            rows[name] = {"n": len(y), **{TARGET_NAMES[k]: int(v) for k, v in counts.items()}}
        return pd.DataFrame(rows).T


def make_split(df, feature_cols, seed=SEED):
    X, y = df[feature_cols], df[TARGET_COL].to_numpy()
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=VAL_SIZE / (1 - TEST_SIZE), stratify=y_tr, random_state=seed)
    return Split(X_tr, X_va, X_te, y_tr, y_va, y_te)


def feature_lists(engineered=True, include_leaky=False):
    numeric = NUMERIC_FEATURES + ORDINAL_FEATURES + (ENGINEERED if engineered else [])
    nominal = NOMINAL_FEATURES + (LEAKY_FEATURES if include_leaky else [])
    return numeric, nominal


def build_preprocessor(numeric, nominal, onehot=True):
    if onehot:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=20)
    else:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())]), numeric),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=-99)), ("encode", encoder)]), nominal),
    ], verbose_feature_names_out=False)


def prepare(engineered=True, include_leaky=False):
    df, meta = load_raw()
    df = engineer_features(clean_sentinels(select_youth(df), meta))
    for col in NOMINAL_FEATURES + LEAKY_FEATURES:
        df[col] = df[col].astype("object")
    numeric, nominal = feature_lists(engineered, include_leaky)
    return make_split(df, numeric + nominal), numeric, nominal, meta, df

### 0.3 Evaluation

Metrics, plots and the experiment log. Accuracy is misleading here because the classes are unbalanced,
so every run also reports balanced accuracy, macro-F1 and one-vs-rest ROC-AUC. Each scikit-learn run
also writes its scores to TensorBoard so the experiment table can be traced to logs.

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    y_bin = label_binarize(y_true, classes=range(N_CLASSES))
    m = {"accuracy": accuracy_score(y_true, y_pred),
         "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
         "f1_macro": f1_score(y_true, y_pred, average="macro"),
         "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
         "roc_auc_ovr_macro": roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"),
         "pr_auc_macro": average_precision_score(y_bin, y_proba, average="macro")}
    return {k: round(float(v), 4) for k, v in m.items()}


def per_class_report(y_true, y_pred):
    rep = classification_report(y_true, y_pred, target_names=CLASS_LABELS, output_dict=True, zero_division=0)
    return pd.DataFrame(rep).T.round(3)


def plot_confusion(y_true, y_pred, title, path):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, norm, sub in zip(axes, [None, "true"], ["counts", "row-normalised"]):
        cm = confusion_matrix(y_true, y_pred, normalize=norm)
        ConfusionMatrixDisplay(cm, display_labels=SHORT_LABELS).plot(ax=ax, cmap="Blues", values_format=".2f" if norm else "d", colorbar=False)
        ax.set_title(f"{title} ({sub})"); ax.tick_params(axis="x", rotation=30)
    fig.tight_layout(); fig.savefig(path); plt.show()


def plot_roc_pr(y_true, y_proba, title, path):
    y_bin = label_binarize(y_true, classes=range(N_CLASSES))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    for k in range(N_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, k], y_proba[:, k])
        ax1.plot(fpr, tpr, label=f"{SHORT_LABELS[k]} (AUC={auc(fpr, tpr):.2f})")
        prec, rec, _ = precision_recall_curve(y_bin[:, k], y_proba[:, k])
        ax2.plot(rec, prec, label=f"{SHORT_LABELS[k]} (AP={average_precision_score(y_bin[:, k], y_proba[:, k]):.2f}, base={y_bin[:, k].mean():.2f})")
    ax1.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax1.set(xlabel="False positive rate", ylabel="True positive rate", title=f"{title}: one-vs-rest ROC"); ax1.legend(fontsize=7)
    ax2.set(xlabel="Recall", ylabel="Precision", title=f"{title}: precision-recall"); ax2.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(path); plt.show()


def learning_curve_data(estimator, X, y, cv=3):
    sizes, tr, va = learning_curve(estimator, X, y, cv=cv, scoring="f1_macro", n_jobs=1,
                                   train_sizes=np.linspace(0.1, 1.0, 6), random_state=SEED, shuffle=True)
    return {"sizes": sizes.tolist(), "train": tr.mean(1).round(4).tolist(), "val": va.mean(1).round(4).tolist()}


def plot_keras_history(history, title, path):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, key in zip(axes, ["loss", "accuracy"]):
        ax.plot(history[key], label=f"train {key}"); ax.plot(history[f"val_{key}"], label=f"val {key}")
        ax.set(xlabel="epoch", ylabel=key, title=f"{title}: {key}"); ax.legend()
    fig.tight_layout(); fig.savefig(path); plt.show()


def log_experiment(run_id, approach, model, hyperparams, val_metrics, test_metrics, notes="",
                   features="base+engineered", logdir="", train_time_s=None):
    row = {"run_id": run_id, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "approach": approach, "model": model,
           "features": features, "split": "70/15/15 stratified, seed 42", "hyperparams": json.dumps(hyperparams, default=str),
           **{f"val_{k}": v for k, v in val_metrics.items()}, **{f"test_{k}": v for k, v in test_metrics.items()},
           "train_time_s": round(train_time_s, 1) if train_time_s else None, "logdir": logdir, "notes": notes}
    df = pd.read_csv(EXPERIMENTS_CSV) if EXPERIMENTS_CSV.exists() else pd.DataFrame()
    if len(df):
        df = df[df.run_id != run_id]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(EXPERIMENTS_CSV, index=False)


def load_experiments():
    return pd.read_csv(EXPERIMENTS_CSV)


def write_tensorboard_scalars(run_id, metrics, hyperparams):
    logdir = LOGS_DIR / run_id
    writer = tf.summary.create_file_writer(str(logdir))
    with writer.as_default():
        for k, v in metrics.items():
            tf.summary.scalar(k, float(v), step=0)
        tf.summary.text("hyperparams", json.dumps(hyperparams, default=str), step=0)
    writer.flush()
    return str(logdir)

### 0.4 Model helpers

One function fits and scores any scikit-learn model inside a pipeline with the encoder. For Keras:
`tf.data` input pipelines, two Functional-API architectures (one-hot MLP and entity-embedding MLP),
and one training loop with early stopping and TensorBoard logging.

In [ ]:
def fit_and_evaluate(estimator, preprocessor, split, sample_weight=None):
    pipe = Pipeline([("prep", clone(preprocessor)), ("model", clone(estimator))])
    t0 = time.time()
    pipe.fit(split.X_train, split.y_train, **({"model__sample_weight": sample_weight} if sample_weight is not None else {}))
    out = {"pipeline": pipe, "train_time_s": time.time() - t0}
    for name, X, y in [("val", split.X_val, split.y_val), ("test", split.X_test, split.y_test)]:
        proba = pipe.predict_proba(X)
        out[f"{name}_proba"], out[f"{name}_pred"] = proba, proba.argmax(1)
        out[f"{name}_metrics"] = compute_metrics(y, proba.argmax(1), proba)
    return out


def balanced_sample_weight(y):
    counts = np.bincount(y, minlength=N_CLASSES).astype(float)
    return (counts.sum() / (N_CLASSES * counts))[y]


def set_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed); keras.utils.set_random_seed(seed)


def make_dataset(inputs, y, batch_size=256, shuffle=False):
    feats = {k: tf.convert_to_tensor(v) for k, v in inputs.items()} if isinstance(inputs, dict) else tf.convert_to_tensor(inputs, tf.float32)
    ds = tf.data.Dataset.from_tensor_slices((feats, tf.convert_to_tensor(y, tf.int32)))
    if shuffle:
        ds = ds.shuffle(min(len(y), 20_000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def split_embedding_inputs(X_mat, n_numeric, nominal_names):
    out = {"numeric": X_mat[:, :n_numeric].astype(np.float32)}
    for j in range(len(nominal_names)):
        out[f"cat_{j}"] = X_mat[:, n_numeric + j].astype(np.int64) + 1      # shift so unknown (-1) becomes 0
    return out


def dense_block(x, hidden, dropout, batch_norm, l2):
    reg = keras.regularizers.l2(l2) if l2 else None
    for i, units in enumerate(hidden):
        x = layers.Dense(units, kernel_regularizer=reg, name=f"dense_{i}")(x)
        if batch_norm:
            x = layers.BatchNormalization(name=f"bn_{i}")(x)
        x = layers.Activation("relu", name=f"act_{i}")(x)
        if dropout:
            x = layers.Dropout(dropout, name=f"drop_{i}")(x)
    return layers.Dense(N_CLASSES, activation="softmax", name="output")(x)


def build_mlp(n_features, hidden=(256, 128), dropout=0.3, batch_norm=True, l2=0.0, name="mlp"):
    inp = keras.Input(shape=(n_features,), name="features")
    return keras.Model(inp, dense_block(inp, hidden, dropout, batch_norm, l2), name=name)


def build_embedding_mlp(n_numeric, cardinalities, nominal_names, hidden=(256, 128), dropout=0.3, batch_norm=True, l2=0.0, name="embedding_mlp"):
    num_in = keras.Input(shape=(n_numeric,), name="numeric")
    inputs, branches = [num_in], [num_in]
    for j, (card, nm) in enumerate(zip(cardinalities, nominal_names)):
        cat_in = keras.Input(shape=(1,), dtype="int64", name=f"cat_{j}")
        dim = int(min(16, max(2, round(1.6 * card ** 0.56))))
        branches.append(layers.Flatten()(layers.Embedding(card + 1, dim, name=f"emb_{nm}")(cat_in)))
        inputs.append(cat_in)
    x = layers.Concatenate(name="concat")(branches)
    return keras.Model(inputs, dense_block(x, hidden, dropout, batch_norm, l2), name=name)


def compile_model(model, lr=1e-3, optimizer="adam", weight_decay=0.0):
    opts = {"adam": keras.optimizers.Adam(lr), "adamw": keras.optimizers.AdamW(lr, weight_decay=weight_decay),
            "sgd": keras.optimizers.SGD(lr, momentum=0.9)}
    model.compile(optimizer=opts[optimizer], loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def class_weights_from(y):
    counts = np.bincount(y, minlength=N_CLASSES).astype(float)
    return {i: float(w) for i, w in enumerate(counts.sum() / (N_CLASSES * counts))}


def train(model, train_ds, val_ds, run_id, epochs=100, patience=10, class_weight=None):
    logdir = LOGS_DIR / run_id
    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
                 keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, patience // 3), min_lr=1e-6),
                 keras.callbacks.TensorBoard(log_dir=str(logdir))]
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, class_weight=class_weight, verbose=0)
    history = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
    return history, {"epochs_run": len(history["loss"]), "best_epoch": int(np.argmin(history["val_loss"])) + 1,
                     "train_time_s": time.time() - t0, "logdir": str(logdir)}

In [ ]:
set_seeds(SEED)
for d in [PROCESSED_DIR, MODELS_DIR, LOGS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)
if EXPERIMENTS_CSV.exists():
    EXPERIMENTS_CSV.unlink()
print("Seed", SEED, "| raw files:", [p.name for p in RAW_DIR.glob(RAW_GLOB)])

## 1. The data

**Source.** Rwanda Labour Force Survey public-use microdata, National Institute of Statistics of Rwanda
(NISR), 2024 round (catalogue 114) and 2025 round (catalogue 125). It is Rwanda's official labour
survey: a stratified two-stage probability sample of households, four quarters pooled per year. The
files need a free NISR account, so they are shared through Drive and never committed.

**Why this data.** It is the official survey for exactly this question, it is nationally
representative, and it has the variables a career programme can act on: education, TVET, disability,
migration, household role, unpaid work. Each round has about 100,000 people, 27,000 of them aged 16–30.

**Who.** People aged 16–30, NISR's definition of "young", with a valid labour force status.

**What we predict.** `status1`, the ILO labour force status NISR computes: employed (any paid work last
week), unemployed (no work, looked for work in the last four weeks, available), or outside the labour
force (everyone else).

In [ ]:
df_raw, meta = load_raw()
print("Rounds:", sorted(df_raw["source_file"].unique()), "| rows (all ages):", f"{len(df_raw):,}")
ages = pd.to_numeric(df_raw["A04"], errors="coerce")
print(f"Age {int(ages.min())}–{int(ages.max())}, median {ages.median():.0f}; share aged 16–30: {ages.between(16, 30).mean():.1%}")
df_raw.groupby("source_file").size().rename("persons").to_frame().T

In [ ]:
youth = select_youth(df_raw)
print("People aged 16–30 with a labour force status:", f"{len(youth):,}")

counts = youth[TARGET_COL].map(TARGET_NAMES).value_counts()
fig, ax = plt.subplots(figsize=(6, 2.6))
sns.barplot(x=counts.values, y=counts.index, ax=ax, color="#4C72B0")
for i, v in enumerate(counts.values):
    ax.text(v + 100, i, f"{v:,} ({v/len(youth):.0%})", va="center")
ax.set(xlabel="people", ylabel="", title="Labour force status, ages 16–30")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig01_target_distribution.png"); plt.show()

w = pd.to_numeric(youth["weight2"], errors="coerce")
weighted = youth.groupby(youth[TARGET_COL].map(TARGET_NAMES)).apply(lambda g: w.loc[g.index].sum()) / w.sum()
pd.DataFrame({"unweighted share": counts / len(youth), "weighted share": weighted}).round(3)

Half are employed, four in ten are outside the labour force, and one in ten is unemployed. The small
class is the one that matters most, so accuracy alone would reward a model that never predicts it.
The survey-weighted shares are almost the same as the raw ones, so I model the raw sample.

### 1.1 A first look

How status varies with the things a programme can act on. Codes are turned into labels for plotting
only.

In [ ]:
lab = lambda col: pd.to_numeric(youth[col], errors="coerce").map(meta.variable_value_labels[col])
plot_df = pd.DataFrame({"status": youth[TARGET_COL].map(TARGET_NAMES), "sex": lab("A01"), "residence": lab("Code_UR"),
                        "province": lab("province"), "education": lab("B02A"), "tvet": lab("TVT"), "studying": lab("B01"),
                        "age group": lab("age3_16_30")})
edu_order = ["None", "Pre-primary", "Primary education", "Lower secondary education", "Upper secondary education", "Tertiary education"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), ["sex", "residence", "province", "education", "tvet", "studying"]):
    ct = pd.crosstab(plot_df[col], plot_df["status"], normalize="index")
    if col == "education":
        ct = ct.reindex([e for e in edu_order if e in ct.index])
    ct.plot(kind="barh", stacked=True, ax=ax, colormap="viridis", legend=(col == "sex"))
    ax.set(title=f"Status by {col}", xlabel="share", ylabel="")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig02_eda_bivariate.png"); plt.show()

In [ ]:
ct = pd.crosstab([plot_df["sex"], plot_df["age group"]], plot_df["status"], normalize="index").round(3)
ct["n"] = plot_df.groupby(["sex", "age group"]).size()
ct

In [ ]:
youth_clean = clean_sentinels(youth, meta)
miss = youth_clean[NUMERIC_FEATURES + ORDINAL_FEATURES + NOMINAL_FEATURES].isna().mean().sort_values(ascending=False)
miss.head(10).round(3).rename("share missing (before skip-pattern handling)").to_frame().T

Three things to keep in mind. The blanks are mostly **skip patterns**, not unknowns: TVET duration and
completion are blank for the 80% who never took a course, hours of chores are blank for people who did
none, and the household-head flag is blank for non-heads — so blanks become zeros or their own
category. Students are almost all outside the labour force, so the 16–19 group looks very different
from the 25–30 group. And women and rural residents shift towards *outside the labour force*, not
towards unemployment.

## 2. Preprocessing and feature engineering

Everything here is fitted on the training split only.

* **No leakage.** NISR builds the target from sections C (work last week, job search), D (main job),
  F (previous job, income) and H (own-use farming), and from derived indicators such as the
  unemployment rate, NEET or hours. None of those are inputs. Only things fixed *before* the outcome
  are used. Experiment E09 adds two of the forbidden items back to show what leakage looks like.
* **"Don't know" codes** (9, 98, …) are found from the value labels in the Stata file, not hard-coded.
* **Skip patterns** become zeros (hours, flags) or an explicit "none" category (TVET items, migration).
* **Features.** 13 numeric, 9 ordered scales treated as numeric, 22 categorical (province, district,
  urban/rural, sex, household role, marital status, studying, TVET items, field of study, lighting,
  migration, age group, survey year), plus 10 engineered ones: `care_hours`, `unpaid_work_hours`,
  `any_disability`, `severe_disability`, `tertiary`, `tvet_completed`, `is_student`, `schooling_gap`
  (years of school missed relative to age), `never_moved`, `married_female`.
* **Encoding.** Numeric: median fill with a missing flag, then standardise. Categorical: missing becomes
  a category, then one-hot (rare levels pooled) — or integer codes for the embedding network.
* **Split.** 70 / 15 / 15, stratified, seed 42. Validation picks models and stops training; the test set
  is used once per final model.

In [ ]:
split, numeric, nominal, meta, youth_fe = prepare()
print(f"{len(numeric)} numeric features, {len(nominal)} categorical features")
display(split.summary())
prep = build_preprocessor(numeric, nominal)
print("Encoded matrix:", prep.fit_transform(split.X_train).shape)
youth_fe[ENGINEERED].describe().T[["mean", "std", "min", "max"]].round(3)

In [ ]:
youth_fe.to_parquet(PROCESSED_DIR / "youth_16_30_clean.parquet", index=False)
json.dump({"numeric": numeric, "nominal": nominal}, open(PROCESSED_DIR / "feature_lists.json", "w"), indent=2)

## 3. Traditional machine learning

One helper runs each experiment: fit the pipeline, score on validation and test, log the row and
write the scores to TensorBoard. The order is deliberate: linear baseline, then feature engineering,
then more flexible models, tuning, class weights, and finally the leakage test.

In [ ]:
def run_ml(run_id, name, estimator, split_, numeric_, nominal_, hyperparams, notes, features="base+engineered", sample_weight=None):
    res = fit_and_evaluate(estimator, build_preprocessor(numeric_, nominal_), split_, sample_weight)
    logdir = write_tensorboard_scalars(run_id, {f"val_{k}": v for k, v in res["val_metrics"].items()}, hyperparams)
    log_experiment(run_id, "traditional ML", name, hyperparams, res["val_metrics"], res["test_metrics"],
                   notes=notes, features=features, logdir=logdir, train_time_s=res["train_time_s"])
    print(f"[{run_id}] {name}: val {res['val_metrics']} ({res['train_time_s']:.0f}s)")
    return res

ml_results = {}

### E01 — Logistic regression on the raw variables
The simplest baseline: linear, fast, easy to read.

In [ ]:
split_raw, numeric_raw, nominal_raw, _, _ = prepare(engineered=False)
ml_results["E01"] = run_ml("E01", "logistic_regression", LogisticRegression(max_iter=3000, C=1.0, random_state=SEED),
                           split_raw, numeric_raw, nominal_raw, {"C": 1.0, "penalty": "l2"}, "Linear baseline, raw variables", features="base only")

### E02 — Logistic regression + engineered features
Same model; the only change is the ten engineered features.

In [ ]:
ml_results["E02"] = run_ml("E02", "logistic_regression", LogisticRegression(max_iter=3000, C=1.0, random_state=SEED),
                           split, numeric, nominal, {"C": 1.0, "penalty": "l2"}, "E01 + engineered features")

**What it shows.** The engineered features add almost nothing to the linear model (macro-F1 0.498 →
0.499). Most of them restate single columns (`is_student` is B01, `tertiary` is B02A). Their value, if
any, comes from interactions, which a linear model cannot use.

### E03 — Random forest, default settings
300 unpruned trees. Trees can learn interactions such as age × studying × sex.

In [ ]:
ml_results["E03"] = run_ml("E03", "random_forest", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
                           split, numeric, nominal, {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 1}, "Default forest")

### E04 — Random forest, tuned
Randomised search (12 settings, 3-fold CV, macro-F1) over depth, leaf size, feature fraction and tree count.

In [ ]:
rf_space = {"model__max_depth": [8, 12, 16, 24, None], "model__min_samples_leaf": [1, 2, 5, 10, 20],
            "model__max_features": ["sqrt", 0.2, 0.4], "model__n_estimators": [300, 500]}
rf_search = RandomizedSearchCV(Pipeline([("prep", build_preprocessor(numeric, nominal)), ("model", RandomForestClassifier(n_jobs=-1, random_state=SEED))]),
                               rf_space, n_iter=12, cv=3, scoring="f1_macro", random_state=SEED, n_jobs=1)
t0 = time.time(); rf_search.fit(split.X_train, split.y_train)
rf_best = {k.replace("model__", ""): v for k, v in rf_search.best_params_.items()}
print(f"search {time.time()-t0:.0f}s | best: {rf_best} | CV macro-F1 {rf_search.best_score_:.4f}")
ml_results["E04"] = run_ml("E04", "random_forest", RandomForestClassifier(n_jobs=-1, random_state=SEED, **rf_best),
                           split, numeric, nominal, {**rf_best, "cv_f1_macro": round(rf_search.best_score_, 4)}, "RandomizedSearchCV(12, cv=3)")

**What it shows.** Trees beat the linear model by four macro-F1 points, so the signal is in
interactions. Tuning adds 0.4 points; the search picked unlimited depth and one sample per leaf, so
variance is being controlled by averaging 500 trees, not by pruning.

### E05 — XGBoost
Gradient boosting with a conservative setting: 400 trees, learning rate 0.05, depth 6, 80% row and column sampling.

In [ ]:
xgb_params = {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 6, "subsample": 0.8, "colsample_bytree": 0.8}
ml_results["E05"] = run_ml("E05", "xgboost", XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", device="cpu", random_state=SEED, n_jobs=-1, **xgb_params),
                           split, numeric, nominal, xgb_params, "Hand-set boosting")

### E06 — LightGBM, tuned
Faster boosting, so a wider search (15 settings) over leaves, learning rate, trees, leaf size, L2 and sampling.

In [ ]:
lgb_space = {"model__num_leaves": [15, 31, 63, 127], "model__learning_rate": [0.02, 0.05, 0.1], "model__n_estimators": [200, 400, 800],
             "model__min_child_samples": [10, 20, 50], "model__reg_lambda": [0.0, 1.0, 5.0], "model__colsample_bytree": [0.5, 0.8, 1.0], "model__subsample": [0.7, 1.0]}
lgb_search = RandomizedSearchCV(Pipeline([("prep", build_preprocessor(numeric, nominal)), ("model", LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1))]),
                                lgb_space, n_iter=15, cv=3, scoring="f1_macro", random_state=SEED, n_jobs=1)
t0 = time.time(); lgb_search.fit(split.X_train, split.y_train)
lgb_best = {k.replace("model__", ""): v for k, v in lgb_search.best_params_.items()}
print(f"search {time.time()-t0:.0f}s | best: {lgb_best} | CV macro-F1 {lgb_search.best_score_:.4f}")
ml_results["E06"] = run_ml("E06", "lightgbm", LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1, **lgb_best),
                           split, numeric, nominal, {**lgb_best, "cv_f1_macro": round(lgb_search.best_score_, 4)}, "RandomizedSearchCV(15, cv=3)")
pd.DataFrame(lgb_search.cv_results_)[["mean_test_score", "std_test_score", "params"]].sort_values("mean_test_score", ascending=False).head(5)

**What it shows.** LightGBM's best setting uses wide trees (127 leaves), learning rate 0.1 and strong
L2. It edges the forest on macro-F1 (0.548 vs 0.543) but XGBoost ranks best on ROC-AUC (0.819): the
models trade ranking quality against the hard boundary around the unemployed class.

### E07 — SVM with an RBF kernel
Kernel SVMs scale badly with rows (and `probability=True` adds an internal 5-fold fit), so it trains on
a stratified 12,000-row subsample of the training set and is scored on the full validation and test
sets. The size is recorded in the table.

In [ ]:
SVM_MAX_ROWS = 12000
idx, _ = train_test_split(np.arange(len(split.y_train)), train_size=SVM_MAX_ROWS, stratify=split.y_train, random_state=SEED)
split_svm = Split(split.X_train.iloc[idx], split.X_val, split.X_test, split.y_train[idx], split.y_val, split.y_test)
ml_results["E07"] = run_ml("E07", "svm_rbf", SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=SEED),
                           split_svm, numeric, nominal, {"C": 1.0, "kernel": "rbf", "gamma": "scale", "train_rows": SVM_MAX_ROWS}, f"RBF SVM on {SVM_MAX_ROWS} rows")

**What it shows.** The SVM only matches the linear baseline (macro-F1 0.500) and took 67 s on a third
of the data — about ten times the cost of boosting. On a sparse one-hot matrix the RBF distance is a
poor measure of similarity.

### E08 — Tuned LightGBM with class weights
Inverse-frequency weights make each unemployed person count as much as five employed ones.

In [ ]:
ml_results["E08"] = run_ml("E08", "lightgbm", LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1, **lgb_best),
                           split, numeric, nominal, {**lgb_best, "class_weight": "balanced"}, "E06 + balanced sample weights",
                           sample_weight=balanced_sample_weight(split.y_train))

**What it shows.** Macro-F1 rises to 0.572 and balanced accuracy to 0.571, while accuracy drops from
0.717 to 0.695: the model now predicts *unemployed* far more often. For a programme that wants to find
job-seekers that is the better trade, so E08 is the final traditional model.

### E09 — Leakage test (not a candidate)
Adds `C01` (paid work last week) and `C19` (looked for work last month), two of the questions NISR uses
to define the target.

In [ ]:
split_leak, numeric_leak, nominal_leak, _, _ = prepare(engineered=True, include_leaky=True)
ml_results["E09"] = run_ml("E09", "lightgbm", LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1, **lgb_best),
                           split_leak, numeric_leak, nominal_leak, {**lgb_best, "features": "+C01 +C19"}, "Leakage test, not a candidate", features="base+engineered+C01+C19")

**What it shows.** 99% accuracy from two questions. This is what a pipeline that used sections C–H
would report — it would be re-computing the definition, not predicting. It also sets the scale for the
honest numbers above.

### 3.1 Learning curves
Macro-F1 against training size (3-fold CV) for the linear model, the tuned forest and the tuned
LightGBM. A big gap between train and validation means variance; two low curves that meet means bias.

In [ ]:
lc = {name: learning_curve_data(ml_results[rid]["pipeline"], split.X_train, split.y_train)
      for rid, name in [("E02", "E02 logistic regression"), ("E04", "E04 random forest (tuned)"), ("E06", "E06 lightgbm (tuned)")]}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (name, d) in zip(axes, lc.items()):
    ax.plot(d["sizes"], d["train"], "o-", label="train"); ax.plot(d["sizes"], d["val"], "o-", label="cross-val")
    ax.set(title=name, xlabel="training rows", ylabel="macro-F1"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig03_learning_curves_ml.png"); plt.show()
pd.DataFrame({k: {"train": d["train"][-1], "cross-val": d["val"][-1], "gap": round(d["train"][-1] - d["val"][-1], 3)} for k, d in lc.items()}).T

**What it shows.** Logistic regression's two curves meet at about 0.50 with a gap of 0.006: bias —
more data will not help it. The ensembles score 1.0 on training data and about 0.53 in cross-validation:
variance, which the tuning (leaf size, L2, subsampling) only partly closed. Every model stops near 0.55,
which points to what the inputs cannot tell us: when someone searched for work and what the local job
market looks like.

## 4. Deep learning

All networks use the Functional API and `tf.data` pipelines (shuffle, batch, prefetch), stop early on
validation loss (patience 10, best weights kept) and log to TensorBoard under `logs/<run_id>`. Two
inputs are compared: the same one-hot matrix as above, and integer codes fed to entity embeddings.

In [ ]:
X_tr, X_va, X_te = (prep.transform(X).astype(np.float32) for X in (split.X_train, split.X_val, split.X_test))
n_features = X_tr.shape[1]

prep_emb = build_preprocessor(numeric, nominal, onehot=False)
M_tr = prep_emb.fit_transform(split.X_train); M_va, M_te = prep_emb.transform(split.X_val), prep_emb.transform(split.X_test)
n_numeric = M_tr.shape[1] - len(nominal)
cardinalities = [len(c) for c in prep_emb.named_transformers_["cat"].named_steps["encode"].categories_]
emb_tr, emb_va, emb_te = (split_embedding_inputs(M, n_numeric, nominal) for M in (M_tr, M_va, M_te))
print("one-hot features:", n_features, "| numeric block:", n_numeric, "| category sizes:", cardinalities)

def run_dl(run_id, model, hyperparams, notes, inputs="dense", batch_size=256, lr=1e-3, optimizer="adam",
           weight_decay=0.0, class_weight=None, epochs=100, patience=10):
    set_seeds(SEED)
    data = (X_tr, X_va, X_te) if inputs == "dense" else (emb_tr, emb_va, emb_te)
    tr, va, te = (make_dataset(X, y, batch_size, shuffle=s) for X, y, s in zip(data, (split.y_train, split.y_val, split.y_test), (True, False, False)))
    compile_model(model, lr, optimizer, weight_decay)
    history, info = train(model, tr, va, run_id, epochs, patience, class_weight)
    p_va, p_te = model.predict(va, verbose=0), model.predict(te, verbose=0)
    val_m, test_m = compute_metrics(split.y_val, p_va.argmax(1), p_va), compute_metrics(split.y_test, p_te.argmax(1), p_te)
    hp = {**hyperparams, "inputs": inputs, "batch_size": batch_size, "lr": lr, "optimizer": optimizer,
          "params": model.count_params(), "epochs_run": info["epochs_run"], "best_epoch": info["best_epoch"]}
    log_experiment(run_id, "deep learning", model.name, hp, val_m, test_m, notes=notes, logdir=info["logdir"], train_time_s=info["train_time_s"])
    plot_keras_history(history, run_id, FIGURES_DIR / f"lc_{run_id}.png")
    print(f"[{run_id}] val {val_m} | epochs {info['epochs_run']} (best {info['best_epoch']}) | {info['train_time_s']:.0f}s")
    return {"model": model, "history": history, "val_metrics": val_m, "test_metrics": test_m, "val_proba": p_va, "test_proba": p_te, "info": info}

dl_results = {}

### D01 — Plain MLP (128–128), no regularisation
The starting point. Expect it to overfit quickly.

In [ ]:
dl_results["D01"] = run_dl("D01", build_mlp(n_features, (128, 128), dropout=0.0, batch_norm=False, name="mlp_plain"),
                           {"hidden": (128, 128), "dropout": 0.0, "batch_norm": False}, "No regularisation")

**What it shows.** Validation loss bottoms out at epoch 5 and then rises while training loss keeps
falling: overfitting. Early stopping kept the epoch-5 weights.

### D02 — MLP (256–128) with dropout 0.3 and batch normalisation

In [ ]:
dl_results["D02"] = run_dl("D02", build_mlp(n_features, (256, 128), dropout=0.3, batch_norm=True, name="mlp_reg"),
                           {"hidden": (256, 128), "dropout": 0.3, "batch_norm": True}, "Dropout + batch norm")

**What it shows.** The minimum moves to epoch 17 and the loss is lower; macro-F1 goes from 0.521 to 0.536, the level of the untuned forest.

### D03 — Bigger network (512–256–128, dropout 0.4)
Does more capacity help once regularised?

In [ ]:
dl_results["D03"] = run_dl("D03", build_mlp(n_features, (512, 256, 128), dropout=0.4, batch_norm=True, name="mlp_deep"),
                           {"hidden": (512, 256, 128), "dropout": 0.4, "batch_norm": True}, "Capacity test")

**What it shows.** Three times the parameters, same validation score (0.531). The data, not the model size, is the limit.

### D04 — Learning rate: 10⁻² and 10⁻⁴ (D02 architecture)

In [ ]:
dl_results["D04a"] = run_dl("D04a", build_mlp(n_features, (256, 128), dropout=0.3, batch_norm=True, name="mlp_reg_lr1e-2"),
                            {"hidden": (256, 128), "dropout": 0.3, "batch_norm": True}, "D02, lr 1e-2", lr=1e-2)
dl_results["D04b"] = run_dl("D04b", build_mlp(n_features, (256, 128), dropout=0.3, batch_norm=True, name="mlp_reg_lr1e-4"),
                            {"hidden": (256, 128), "dropout": 0.3, "batch_norm": True}, "D02, lr 1e-4", lr=1e-4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for rid, lr_ in [("D04a", "1e-2"), ("D02", "1e-3"), ("D04b", "1e-4")]:
    h = dl_results[rid]["history"]
    axes[0].plot(h["val_loss"], label=f"{rid} lr={lr_}"); axes[1].plot(h["loss"], label=f"{rid} lr={lr_}")
axes[0].set(title="validation loss", xlabel="epoch", ylabel="loss"); axes[1].set(title="training loss", xlabel="epoch")
axes[0].legend(); axes[1].legend(); fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig04_lr_sweep.png"); plt.show()

**What it shows.** At 10⁻² the validation loss jumps around and stops early with a worse minimum; at
10⁻⁴ it is still falling at epoch 100; 10⁻³ gets to the lowest loss fastest. All three end within 1.5
macro-F1 points, so the learning rate changes the path more than the destination.

### D05 — Entity-embedding MLP
Each categorical variable gets its own small learned vector (size about 1.6·n^0.56, max 16) instead of
one-hot columns. This lets the network learn that two districts are similar.

In [ ]:
dl_results["D05"] = run_dl("D05", build_embedding_mlp(n_numeric, cardinalities, nominal, (256, 128), dropout=0.3, name="embedding_mlp"),
                           {"hidden": (256, 128), "dropout": 0.3, "batch_norm": True, "embeddings": True}, "Entity embeddings", inputs="embedding")

**What it shows.** Same score as one-hot (0.537 vs 0.536) with fewer parameters but slower convergence. With at most 50 levels per variable, one-hot is already compact.

### D06 — Embedding MLP with class weights
The deep-learning version of E08.

In [ ]:
dl_results["D06"] = run_dl("D06", build_embedding_mlp(n_numeric, cardinalities, nominal, (256, 128), dropout=0.3, name="embedding_mlp_cw"),
                           {"hidden": (256, 128), "dropout": 0.3, "batch_norm": True, "class_weight": "balanced"}, "D05 + class weights",
                           inputs="embedding", class_weight=class_weights_from(split.y_train))

**What it shows.** Balanced accuracy 0.612 and unemployed recall 0.55 on test, at the cost of accuracy
(0.622). Best deep run by macro-F1 (0.567), the same effect class weights had on LightGBM.

### D07 — AdamW with weight decay and L2, batch 512
A different route to regularisation than dropout.

In [ ]:
dl_results["D07"] = run_dl("D07", build_mlp(n_features, (256, 128), dropout=0.2, batch_norm=True, l2=1e-4, name="mlp_adamw"),
                           {"hidden": (256, 128), "dropout": 0.2, "batch_norm": True, "l2": 1e-4}, "AdamW + L2, batch 512",
                           optimizer="adamw", weight_decay=1e-4, batch_size=512)

### D08 — SGD with momentum (lr 0.05) on the D02 architecture

In [ ]:
dl_results["D08"] = run_dl("D08", build_mlp(n_features, (256, 128), dropout=0.3, batch_norm=True, name="mlp_sgd"),
                           {"hidden": (256, 128), "dropout": 0.3, "batch_norm": True}, "SGD momentum 0.9", optimizer="sgd", lr=0.05, epochs=150, patience=15)

**What it shows.** AdamW (0.544) sits between D02 and D06 with the smoothest validation curve; SGD needs
almost twice the epochs and ends slightly lower. No optimiser or regulariser gets a network past about
0.54 without class weights.

## 5. Results

### 5.1 Experiment table
All 18 runs with their settings, validation and test scores and the TensorBoard log that backs each
row. Also saved to `reports/experiments.csv` and `reports/experiment_table.md`.

In [ ]:
exp = load_experiments()
cols = ["run_id", "approach", "model", "features", "val_accuracy", "val_balanced_accuracy", "val_f1_macro", "val_roc_auc_ovr_macro",
        "test_accuracy", "test_balanced_accuracy", "test_f1_macro", "test_roc_auc_ovr_macro", "train_time_s", "notes"]
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 200)
exp[cols].to_markdown(ROOT / "reports" / "experiment_table.md", index=False)
exp[["run_id", "hyperparams"]].to_markdown(ROOT / "reports" / "experiment_hyperparams.md", index=False)
exp[cols]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
exp[exp.run_id != "E09"].set_index("run_id")[["val_f1_macro", "val_balanced_accuracy", "val_roc_auc_ovr_macro"]].plot(kind="bar", ax=ax)
ax.set(ylabel="validation score", title="Validation scores by experiment (leakage test E09 left out)", ylim=(0.4, 1.0)); ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig05_experiment_comparison.png"); plt.show()

In [ ]:
candidates = exp[exp.run_id != "E09"]
best_ml_id = candidates[candidates.approach == "traditional ML"].sort_values("val_f1_macro").iloc[-1]["run_id"]
best_dl_id = candidates[candidates.approach == "deep learning"].sort_values("val_f1_macro").iloc[-1]["run_id"]
best_ml, best_dl = ml_results[best_ml_id], dl_results[best_dl_id]
print("Best by validation macro-F1 — traditional:", best_ml_id, "| deep:", best_dl_id)
pd.DataFrame({best_ml_id: best_ml["test_metrics"], best_dl_id: best_dl["test_metrics"]})

### 5.2 Confusion matrices (test set)

In [ ]:
plot_confusion(split.y_test, best_ml["test_pred"], f"{best_ml_id} test", FIGURES_DIR / "fig06_confusion_best_ml.png")
plot_confusion(split.y_test, best_dl["test_proba"].argmax(1), f"{best_dl_id} test", FIGURES_DIR / "fig07_confusion_best_dl.png")
print(best_ml_id); display(per_class_report(split.y_test, best_ml["test_pred"]))
print(best_dl_id); display(per_class_report(split.y_test, best_dl["test_proba"].argmax(1)))

**What it shows.** E08 finds 75% of the employed and 72% of those outside the labour force but only
20% of the unemployed — over half of them are called employed. D06 flips the trade: 55% of the
unemployed found, but 29% of the employed mislabelled. Both rank people almost identically (ROC-AUC
0.79–0.80); neither can separate the unemployed well, because looking for work is not visible in the
inputs.

### 5.3 ROC and precision–recall curves (test set)

In [ ]:
plot_roc_pr(split.y_test, best_ml["test_proba"], best_ml_id, FIGURES_DIR / "fig08_roc_pr_best_ml.png")
plot_roc_pr(split.y_test, best_dl["test_proba"], best_dl_id, FIGURES_DIR / "fig09_roc_pr_best_dl.png")

**What it shows.** ROC-AUC is fine for the two big classes (0.82 and 0.86) but only 0.70 for the
unemployed, and the precision–recall curve is blunt: average precision 0.20 against a base rate of 0.10
— twice chance, no more. ROC-AUC alone would have flattered the model; that is why macro-F1 drove model
selection.

### 5.4 Deep-learning loss curves side by side

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, rid in zip(axes.ravel(), ["D01", "D02", "D03", "D05", "D06", "D07", "D08", "D04a"]):
    h = dl_results[rid]["history"]
    ax.plot(h["loss"], label="train"); ax.plot(h["val_loss"], label="val"); ax.axvline(dl_results[rid]["info"]["best_epoch"] - 1, color="grey", ls=":", lw=0.8)
    ax.set(title=f"{rid} ({dl_results[rid]['model'].name})", xlabel="epoch", ylabel="loss"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig10_dl_learning_curves.png"); plt.show()

### 5.5 Which variables matter?
Permutation importance for the best traditional model: shuffle one raw column at a time and record the
drop in validation macro-F1 (average of 5 repeats).

In [ ]:
pi = permutation_importance(best_ml["pipeline"], split.X_val, split.y_val, scoring="f1_macro", n_repeats=5, random_state=SEED, n_jobs=1)
labels = {c: (meta.column_names_to_labels.get(c) or c)[:45] for c in split.X_val.columns}
imp = pd.DataFrame({"feature": split.X_val.columns, "label": [labels[c] for c in split.X_val.columns],
                    "importance": pi.importances_mean, "std": pi.importances_std}).sort_values("importance", ascending=False)
top = imp.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top["feature"] + " — " + top["label"], top["importance"], xerr=top["std"], color="#4C72B0")
ax.set(xlabel="drop in validation macro-F1 when shuffled", title=f"Permutation importance, {best_ml_id}")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig11_permutation_importance.png"); plt.show()
imp.to_csv(ROOT / "reports" / "permutation_importance.csv", index=False)
imp.head(12).round(4)

**What it shows.** Being a student matters most (0.041), then household role, hours of unpaid work,
district, household size, age, sex and years of schooling. The TVET variables are near the bottom. The
model mostly learns a person's position in life — child at school, spouse doing unpaid work, head of
household — not their skills. For a career programme that is a finding in itself: attending training
says little about *whether* a young person is working.

### 5.6 Where does the model fail?
Test accuracy of the best traditional model by group, the most common confusions, and recall on the
unemployed by sex and residence.

In [ ]:
err = split.X_test.copy()
err["y_true"], err["y_pred"] = split.y_test, best_ml["test_pred"]
err["correct"] = err.y_true == err.y_pred
vl = lambda col: pd.to_numeric(err[col], errors="coerce").map(meta.variable_value_labels[col])
err["sex"], err["residence"], err["province"] = vl("A01"), vl("Code_UR"), vl("province")
err["studying"], err["age group"], err["education"], err["district"] = vl("B01"), vl("age3_16_30"), vl("B02A"), vl("code_dis")

by_group = pd.concat([err.groupby(col)["correct"].agg(accuracy="mean", n="size").rename(index=lambda i: f"{col}: {i}")
                      for col in ["sex", "residence", "province", "education", "studying", "age group"]])
display(by_group.round(3))

cm = confusion_matrix(err.y_true, err.y_pred)
conf = [(TARGET_NAMES[i], TARGET_NAMES[j], int(cm[i, j]), round(cm[i, j] / cm[i].sum(), 3)) for i in range(3) for j in range(3) if i != j]
display(pd.DataFrame(conf, columns=["true", "predicted", "count", "share of true class"]).sort_values("count", ascending=False))

unemp = err[err.y_true == 1]
display(unemp.groupby(["sex", "residence"]).apply(lambda g: pd.Series({"unemployed recall": (g.y_pred == 1).mean(), "n": len(g)})).round(3))

In [ ]:
by_district = err.groupby("district")["correct"].mean().sort_values()
fig, ax = plt.subplots(figsize=(6, 7))
ax.barh(by_district.index, by_district.values, color="#4C72B0"); ax.axvline(err["correct"].mean(), color="k", ls="--", lw=0.8, label="overall")
ax.set(xlabel="test accuracy", title=f"Test accuracy by district, {best_ml_id}"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig12_accuracy_by_district.png"); plt.show()

**What it shows.** Accuracy is 0.74 for men and 0.63 for women, 0.72 urban and 0.66 rural, 0.84 for
students and 0.63 for everyone else; districts range from 0.58 (Gisagara) to 0.74 (Kicukiro). Recall on
the unemployed is lowest for rural women (0.14). The model is good where status is fixed by an
institution (school) and weak where it depends on informal, shifting activity: rural women move between
subsistence farming, care and occasional job search, and the survey's background variables cannot tell
those apart.

## 6. Saving the best models
The best scikit-learn pipeline (encoder + model) with joblib, the best Keras model in `.keras` format,
and the fitted encoders. They are shared through Drive.

In [ ]:
ml_path = MODELS_DIR / f"best_traditional_{best_ml_id}_{exp.set_index('run_id').loc[best_ml_id, 'model']}.joblib"
dl_path = MODELS_DIR / f"best_deep_{best_dl_id}_{best_dl['model'].name}.keras"
joblib.dump(best_ml["pipeline"], ml_path)
best_dl["model"].save(dl_path)
joblib.dump({"dense": prep, "embedding": prep_emb, "numeric": numeric, "nominal": nominal, "n_numeric": n_numeric}, MODELS_DIR / "preprocessors.joblib")
print("saved:", ml_path.name, f"({ml_path.stat().st_size/1e6:.1f} MB) |", dl_path.name, f"({dl_path.stat().st_size/1e6:.1f} MB)")

reloaded = joblib.load(ml_path)
assert np.allclose(reloaded.predict_proba(split.X_test[:50]), best_ml["test_proba"][:50], atol=1e-6)
keras.models.load_model(dl_path)
print("reload check passed")

## 7. Evidence of runs
Every experiment wrote to `logs/<run_id>`: per-epoch curves for the Keras runs, final scores and
settings for the scikit-learn runs. Uncomment the magic to browse them. The logs are zipped into
`reports/`, and on Colab everything is copied to Drive.

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir logs

In [ ]:
archive = shutil.make_archive(str(ROOT / "reports" / "tensorboard_logs"), "zip", LOGS_DIR)
print("TensorBoard logs archived:", Path(archive).name, "| runs:", sorted(p.name for p in LOGS_DIR.iterdir()))
if IN_COLAB:
    out = drive_dir / "outputs"
    for src in ["reports", "models", "logs", "notebooks"]:
        shutil.copytree(src, out / src, dirs_exist_ok=True)
    print("Copied outputs to", out)

## 8. Conclusions

* **Eighteen logged experiments** on 53,503 young Rwandans. The best traditional model (E08,
  LightGBM with class weights) reaches test macro-F1 0.556, balanced accuracy 0.555 and ROC-AUC 0.79;
  the best network (D06, embedding MLP with class weights) reaches 0.554, 0.597 and 0.80. A tie on
  macro-F1: the network finds more of the unemployed, the boosting model is more accurate overall and
  trains ten times faster.
* **Feature engineering** barely moved the linear model; the trees find the interactions themselves.
  Only `is_student` and `unpaid_work_hours` made the top of the importance list.
* **Learning curves** show bias for the linear model, variance for the ensembles, and a shared ceiling
  near 0.55 that no size, optimiser or learning rate crossed — the limit is the inputs, not the models.
* **Class imbalance** is the central difficulty: ROC-AUC 0.70 and average precision 0.20 for the
  unemployed. Class weights were the only change that moved that class, by trading accuracy for recall.
* **Errors** concentrate on rural women (unemployed recall 0.14), whose situations are informal and
  shifting; students (84% accuracy) are easy.
* **The leakage test** (99% accuracy) shows why the honest numbers look modest.
* **Next steps:** tune decision thresholds per class, model in/out of the labour force first and then
  employed/unemployed, use the survey's quarterly re-interviews to predict *transitions*, and judge
  training programmes by job quality rather than by employment status.